In [112]:
# AUTOGENERATED! DO NOT EDIT! File to edit: ../../nbs/dvc.experiment.ipynb.

# %% auto 0
__all__ = ["parse_params", "parse_metrics", "parse_experiment", "parse_experiments", "load_experiments"]

# %% ../../nbs/dvc.experiment.ipynb 3
import json
from typing import Generator


# %% ../../nbs/dvc.experiment.ipynb 4
def parse_params(record):
    params_node = record.get("data", {}).get("params", {})
    params = {}
    for k, v in params_node.items():
        params.update(v.get("data", {}))
    return params


def parse_metrics(record):
    metrics_node = record.get("data", {}).get("metrics", {})
    metrics = {}
    for k, v in metrics_node.items():
        metrics.update(v.get("data", {}))
    return metrics


def parse_experiment(record):
    return {
        "id": record["rev"],
        "name": record["name"],
        "params": parse_params(record),
        "metrics": parse_metrics(record),
    }


def parse_experiments(data: list[dict]) -> Generator[dict, None, None]:
    for node in data:
        if node.get("error"):
            continue
        commit = node.get("rev")
        if experiments := (node.get("experiments") or []):
            for experiment in experiments:
                for rev in experiment.get("revs") or []:
                    if not rev.get("error"):
                        yield {"commit": commit, **parse_experiment(rev)}
        else:
            yield {"commit": commit, **parse_experiment(node)}


def load_experiments(json_filepath):
    with open(json_filepath, "r") as f:
        data = json.load(f)
    return list(parse_experiments(data))


In [113]:
import itertools
import json
from pathlib import Path

import pandas as pd

In [114]:
def sorted_tuple(x):
    return tuple(sorted(x))

In [115]:
def jprint(obj):
    print(json.dumps(obj, indent=2))

In [116]:
filepaths = list(Path("../tmp/experiments/").glob("*.json"))
experiments = [exp for fp in filepaths for exp in load_experiments(fp)]
print(f"{len(experiments)} experiments")
jprint(next(iter(experiments), None))

10 experiments
{
  "commit": "workspace",
  "id": "workspace",
  "name": null,
  "params": {
    "dataset": {
      "path": "bdsaglam/musique-mini",
      "name": "answerable",
      "split": "validation"
    },
    "model": {
      "name": "Qwen/Qwen2.5-1.5B-Instruct",
      "temperature": 0.3,
      "top_p": 0.95
    },
    "retriever": {
      "name": "semantic",
      "top_k": 2
    },
    "run": 1
  },
  "metrics": {
    "exact_match": 0.0033333333333333335,
    "f1": 0.014063492063492061
  }
}


In [117]:
df = pd.json_normalize(experiments)
print(f"{len(df)} experiments before preprocessing")
df.head()

10 experiments before preprocessing


,commit,id,name,params.dataset.path,params.dataset.name,params.dataset.split,params.model.name,params.model.temperature,params.model.top_p,params.retriever.name,params.retriever.top_k,params.run,metrics.exact_match,metrics.f1
0,workspace,workspace,None,bdsaglam/musique-mini,answerable,validation,Qwen/Qwen2.5-1.5B-Instruct,0.3,0.95,semantic,2,1,0.003333,0.014063
1,8024c394ded6e22903738cef038b5a11b8cceace,86e5dbccd1a516dafa43b118b0df0cf08575e185,warty-cart,bdsaglam/musique-mini,answerable,validation,bdsaglam/Qwen2.5-1.5B-Instruct-ragent-musique,0.0,0.95,hybrid,2,1,0.130000,0.184990
2,8024c394ded6e22903738cef038b5a11b8cceace,55e16ffa8d8a7b9dea7fdeddb85b1a23da125010,trade-hobo,bdsaglam/musique-mini,answerable,validation,Qwen/Qwen2.5-7B-Instruct,0.3,0.95,hybrid,1,1,0.176667,0.264865
3,8024c394ded6e22903738cef038b5a11b8cceace,382abdb8fc38553c30a117eadb9f5c315c68674f,prosy-matt,bdsaglam/musique-mini,answerable,validation,Qwen/Qwen2.5-1.5B-Instruct,0.3,0.95,lexical,3,1,0.006667,0.011460
4,8024c394ded6e22903738cef038b5a11b8cceace,f5c8b471ebae377cde4fb012e3b77f758de0cba7,gross-nuke,bdsaglam/musique-mini,answerable,validation,Qwen/Qwen2.5-1.5B-Instruct,0.0,0.95,lexical,3,1,0.006667,0.011460


In [118]:
param_cols = [col for col in df.columns if col.startswith("params.")]
metric_cols = [col for col in df.columns if col.startswith("metrics.")]

In [119]:
df.dropna(subset=param_cols + metric_cols, inplace=True, how="any")
df.drop_duplicates(subset=param_cols, inplace=True, keep='last')

print(f"{len(df)} experiments after preprocessing")
df.head()

10 experiments after preprocessing


,commit,id,name,params.dataset.path,params.dataset.name,params.dataset.split,params.model.name,params.model.temperature,params.model.top_p,params.retriever.name,params.retriever.top_k,params.run,metrics.exact_match,metrics.f1
0,workspace,workspace,None,bdsaglam/musique-mini,answerable,validation,Qwen/Qwen2.5-1.5B-Instruct,0.3,0.95,semantic,2,1,0.003333,0.014063
1,8024c394ded6e22903738cef038b5a11b8cceace,86e5dbccd1a516dafa43b118b0df0cf08575e185,warty-cart,bdsaglam/musique-mini,answerable,validation,bdsaglam/Qwen2.5-1.5B-Instruct-ragent-musique,0.0,0.95,hybrid,2,1,0.130000,0.184990
2,8024c394ded6e22903738cef038b5a11b8cceace,55e16ffa8d8a7b9dea7fdeddb85b1a23da125010,trade-hobo,bdsaglam/musique-mini,answerable,validation,Qwen/Qwen2.5-7B-Instruct,0.3,0.95,hybrid,1,1,0.176667,0.264865
3,8024c394ded6e22903738cef038b5a11b8cceace,382abdb8fc38553c30a117eadb9f5c315c68674f,prosy-matt,bdsaglam/musique-mini,answerable,validation,Qwen/Qwen2.5-1.5B-Instruct,0.3,0.95,lexical,3,1,0.006667,0.011460
4,8024c394ded6e22903738cef038b5a11b8cceace,f5c8b471ebae377cde4fb012e3b77f758de0cba7,gross-nuke,bdsaglam/musique-mini,answerable,validation,Qwen/Qwen2.5-1.5B-Instruct,0.0,0.95,lexical,3,1,0.006667,0.011460


In [120]:
for col in param_cols:
    values = list(df[col].unique())
    print(f"- {col}: {values}")
    print()

- params.dataset.path: ['bdsaglam/musique-mini']

- params.dataset.name: ['answerable']

- params.dataset.split: ['validation']

- params.model.name: ['Qwen/Qwen2.5-1.5B-Instruct', 'bdsaglam/Qwen2.5-1.5B-Instruct-ragent-musique', 'Qwen/Qwen2.5-7B-Instruct', 'meta-llama/Llama-3.1-8B-Instruct']

- params.model.temperature: [0.3, 0.0]

- params.model.top_p: [0.95]

- params.retriever.name: ['semantic', 'hybrid', 'lexical']

- params.retriever.top_k: [2, 1, 3]

- params.run: [1]



In [121]:
df.to_json('exps.jsonl', orient='records', lines=True)

## Setup remaining experiments

In [122]:
df = pd.read_json('exps.jsonl', orient='records', lines=True)

In [123]:
def produce_experiment_configs(common_params, varying_params):
    # Generate all possible combinations of parameters
    varying_params = {**common_params, **varying_params}
    keys = varying_params.keys()
    values = varying_params.values()
    for instance in itertools.product(*values):
        yield dict(zip(keys, instance))

In [124]:
def produce_all_experiment_configs(common_params: dict, varying_params_list: list[dict]):
    for params in varying_params_list:
        for exp_config in produce_experiment_configs(common_params, params):
            yield exp_config

In [125]:
common_params = {
    "params.dataset.path": ["bdsaglam/musique-mini"],
    "params.dataset.name": ["answerable"],
    "params.dataset.split": ['"validation"'],
    "params.model.name": [
        "Qwen/Qwen2.5-1.5B-Instruct",
        "Qwen/Qwen2.5-7B-Instruct",
        "meta-llama/Llama-3.1-8B-Instruct",
        "bdsaglam/Qwen2.5-1.5B-Instruct-ragent-musique",
    ],
    "params.model.temperature": [
        # 0.0,
        0.3,
        # 0.5,
        # 0.7,
    ],
    "params.model.top_p": [
        0.95,
    ],
    "params.retriever.name": [
        # "semantic",
        "lexical",
        "hybrid",
    ],
    "params.retriever.top_k": [
        1,
        2,
        3,
    ],
    "params.run": [
        1,
        # 2,
        # 3,
    ],
}

In [126]:
varying_params_list = [
    {
        "params.run": [
            1,
            # 2,
            # 3,
        ],
    }
]

In [127]:
exp_configs = list(produce_all_experiment_configs(common_params, varying_params_list))
target_params = list(exp_configs[0].keys())
print(f"{len(exp_configs)} experiment configurations")
print(target_params)

24 experiment configurations
['params.dataset.path', 'params.dataset.name', 'params.dataset.split', 'params.model.name', 'params.model.temperature', 'params.model.top_p', 'params.retriever.name', 'params.retriever.top_k', 'params.run']


In [128]:
if len(df):
    existing_configs = df[target_params].to_dict(orient="records")
    existing_configs[0]
else:
    existing_configs = []

print("Existing exps:", len(existing_configs))

Existing exps: 10


In [129]:
# find the missing configurations
missing_configs = [
    dict(kv)
    for kv in list(
        {tuple(sorted(config.items())) for config in exp_configs}
        - {tuple(sorted(config.items())) for config in existing_configs}
    )
]
print(f"{len(missing_configs)} missing configurations")

24 missing configurations


In [130]:
def make_command(exp_config):
    lines = ["dvc exp run --queue"]
    for target_param in target_params:
        arg_name = target_param.split(".", 1)[-1]
        arg_value = exp_config[target_param]
        lines.append(f"-S {arg_name}='{arg_value}'")

    command = " \\\n    ".join(lines)
    return command

In [131]:
with open("run.sh", "w") as f:
    f.write("#!/bin/sh\n\n")
    for exp_config in missing_configs:
        f.write(make_command(exp_config))
        f.write("\n\n")

## Inspect